In [1]:
import numpy as np
import pandas as pd
import os
import time
import logging
from scipy.io import *
from sklearn.model_selection import LeaveOneOut
from sklearn.svm import SVC
from mlxtend.evaluate import permutation_test

def getVals(fileOfInterest):
    mat = loadmat(fileOfInterest)
    firingRatesNormed = mat.get('firRates')
    firRatesNormed = np.nan_to_num(firingRatesNormed,nan=0.)
    startArm = mat.get('NorthSouth').flatten()
    behavior = mat.get('EastWest').flatten()
    return firRatesNormed,startArm,behavior

def getShuffleAcc(shufflePath):
    shuffleAcc = []
    for sRoot,sDirs,sFiles in os.walk(shufflePath):
        for sFile in sFiles:
            if sFile.endswith('.mat'):
                shuffleBeingChecked = os.path.join(shufflePath,sFile)
                normFR, sArm, beh = getVals(shuffleBeingChecked)
                looS = LeaveOneOut()
                looS.get_n_splits(normFR)
                for trainInds,testInd in looS.split(normFR):
                    xTrainS,xTestS = normFR[trainInds],normFR[testInd]
                    yTrainS,yTestS = beh[trainInds],beh[testInd]
                    svClassifier = SVC(C=1.0,kernel='rbf',degree=4,gamma='auto')
                    svClassifier.fit(xTrainS,yTrainS)
                    yPred = svClassifier.predict(xTestS)
                    if yPred == yTestS:
                        shuffleAcc.append(1)
                    elif yPred != yTestS:
                        shuffleAcc.append(0)
    return shuffleAcc

def getRealAcc(realValsFile):
    realAcc = []
    normRealFR, sArmReal, behReal = getVals(realValsFile)
    looR = LeaveOneOut()
    looR.get_n_splits(normRealFR)
    for trainInds,testInd in looR.split(normRealFR):
        xTrainR,xTestR = normRealFR[trainInds],normRealFR[testInd]
        yTrainR,yTestR = behReal[trainInds],behReal[testInd]
        svClassifier = SVC(C=1.0,kernel='rbf',degree=4,gamma='auto')
        svClassifier.fit(xTrainR,yTrainR)
        yPred = svClassifier.predict(xTestR)
        if yPred == yTestR:
            realAcc.append(1)
        elif yPred != yTestR:
            realAcc.append(0)
    return realAcc

trueValsP = ['',
            ]

shufflePs = ['',
            ]

saveValsP = ['',
            ]

for z in range(len(trueValsP)):
    realAccVals = getRealAcc(trueValsP[z])
    realAccVal = np.sum(realAccVals)/len(realAccVals)
    shuffleAccVals = getShuffleAcc(shufflePs[z])
    shuffAccVal = np.sum(shuffleAccVals)/len(shuffleAccVals)
    p = permutation_test(realAccVals,shuffleAccVals,method='approximate',num_rounds=1000,seed=0)
    saveDict = {'realAcc':realAccVal,'shuffleAcc':shuffAccVal,'pVal':p}
    var = savemat(saveValsP[z],saveDict)
    print(f'Saved successfully at {saveValsP[z]}')
